# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamzaviour/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

**Lane:** Content Opportunity Scoring (refresh prioritisation). Develop month: `2026-03` (mid-panel). June 2026 is a sealed test month and is never touched here.

Order: 1) contract in plain words, 2) three verification queries, 3) five features, 4) the leakage trap, 5) one named limitation, 6) self-check.

## 1) The contract — five plain-words answers

1. **What one row means.** The warehouse grain is one *page on one day* (`report_date × client × content`). For modelling I roll that up to **one row per page**: its first 14 days of March, as of the decision moment (14 March).
2. **Table(s).** `fact_content_daily_performance`, partition `month=2026-03`. No joins needed here.
3. **Time window.** Features use `2026-03-01 … 2026-03-14`. The outcome window is `2026-03-15 … 2026-03-31`. Nothing from June 2026.
4. **What I rank/predict (label proxy).** "Will this page's daily impressions grow by more than 20% in the second half of the month versus the first half?" (`grows = 1`). A refresh team ranks pages by predicted probability. It is a proxy for opportunity, not opportunity itself.
5. **One thing deliberately excluded.** Anything computed from the outcome window (15–31 March). GA4 columns are also left out of the features, because `ga4_data_available` is three-valued (TRUE / FALSE / NULL) and many rows lack it. Identifiers (client, content) are context for grouping only, never features.

In [ ]:
# Setup — HF_TOKEN must be a Colab Secret (key icon). Never paste a token into a cell.
!pip install -q duckdb scikit-learn pandas pyarrow
import os, duckdb, pandas as pd, numpy as np
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "Add HF_TOKEN as a Colab Secret (plain READ token) and enable notebook access"

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

# Schema check (not one of the three queries): read the real column names instead of guessing.
cols = con.sql(f"DESCRIBE SELECT * FROM {FACT}").df()
print(cols[["column_name", "column_type"]].to_string(index=False))

In [ ]:
# Map the column names this notebook relies on; fail loudly if one is missing.
names = set(cols.column_name)
def pick(*cands):
    for c in cands:
        if c in names: return c
    raise KeyError(f"none of {cands} in schema -> edit this cell using the table printed above")
CONTENT = pick("content_id", "content_hash_id")
CLIENT  = pick("client_id", "client_hash_id")
IMP, CLK, POS = pick("gsc_impressions"), pick("gsc_clicks"), pick("gsc_avg_position")
GSC_OK, GA4_OK = pick("gsc_data_available"), pick("ga4_data_available")
print(CONTENT, CLIENT, IMP, CLK, POS, GSC_OK, GA4_OK)

## 2) Three verification queries (month=2026-03)

### Query 1 — grain: one row really is one page-day

In [ ]:
q1 = con.sql(f"""
    SELECT COUNT(*) AS rows_total,
           COUNT(*) - COUNT(DISTINCT (report_date, {CLIENT}, {CONTENT})) AS duplicate_rows
    FROM {FACT}
""").df()
q1

`duplicate_rows = 0` means `(report_date, client, content)` is the grain.

### Query 2 — slice row count and date span

In [ ]:
q2 = con.sql(f"""
    SELECT COUNT(*) AS rows_in_slice,
           MIN(report_date) AS first_day, MAX(report_date) AS last_day,
           COUNT(DISTINCT report_date) AS days,
           COUNT(DISTINCT {CLIENT}) AS clients,
           COUNT(DISTINCT {CONTENT}) AS pages
    FROM {FACT}
""").df()
q2

### Query 3 — availability: filter with `IS TRUE`, count survivors

In [ ]:
q3 = con.sql(f"""
    SELECT COUNT(*) AS rows_total,
           SUM(({GSC_OK} IS TRUE)::INT) AS gsc_is_true,
           SUM(({GA4_OK} IS TRUE)::INT) AS ga4_is_true,
           SUM(({GA4_OK} IS NULL)::INT) AS ga4_null,
           SUM(({GA4_OK} IS FALSE)::INT) AS ga4_false
    FROM {FACT}
""").df()
q3["gsc_pct"] = (100 * q3.gsc_is_true / q3.rows_total).round(1)
q3["ga4_pct"] = (100 * q3.ga4_is_true / q3.rows_total).round(1)
q3

Rows that survive `gsc_data_available IS TRUE` are my slice. GA4 survives on a smaller share (`ga4_pct`, `ga4_null`), which is why GA4 stays out of the features.

## 3) Five features — each knowable at the decision moment (14 March)

Built only from `2026-03-01 … 2026-03-14` rows with `gsc_data_available IS TRUE`. Pages need ≥ 50 early impressions so a growth ratio means something.

| feature | knowable at the decision moment because… |
|---|---|
| `imp_early` | sum of impressions through 14 Mar — already happened by then |
| `clicks_early` | sum of clicks through 14 Mar — already happened |
| `ctr_early` | clicks ÷ impressions, both through 14 Mar |
| `pos_early` | mean GSC position (ignoring 0/NULL) through 14 Mar |
| `active_days_early` | days with ≥ 1 impression among 1–14 Mar — a count of past days |

In [ ]:
frame = con.sql(f"""
    SELECT {CLIENT} AS client, {CONTENT} AS content,
           SUM(CASE WHEN report_date <= DATE '2026-03-14' THEN {IMP} END)                  AS imp_early,
           SUM(CASE WHEN report_date <= DATE '2026-03-14' THEN {CLK} END)                  AS clicks_early,
           AVG(CASE WHEN report_date <= DATE '2026-03-14' AND {POS} > 0 THEN {POS} END)    AS pos_early,
           SUM(CASE WHEN report_date <= DATE '2026-03-14' AND {IMP} > 0 THEN 1 ELSE 0 END) AS active_days_early,
           SUM(CASE WHEN report_date >  DATE '2026-03-14' THEN {IMP} END)                  AS imp_late  -- OUTCOME WINDOW
    FROM {FACT}
    WHERE {GSC_OK} IS TRUE
    GROUP BY 1, 2
    HAVING SUM(CASE WHEN report_date <= DATE '2026-03-14' THEN {IMP} END) >= 50
""").df()
frame["ctr_early"] = frame.clicks_early / frame.imp_early
# label proxy: daily impressions in 15-31 Mar (17 days) vs 1-14 Mar (14 days) grew by > 20%
frame["grows"] = ((frame.imp_late.fillna(0) / 17) > 1.2 * (frame.imp_early / 14)).astype(int)
FEATS = ["imp_early", "clicks_early", "ctr_early", "pos_early", "active_days_early"]
print(frame.shape, "positive rate:", round(frame.grows.mean(), 3))
frame[FEATS + ["grows"]].head()

## 4) The trap — one label-derived column, on purpose

I add `LEAK_imp_late` (impressions in the outcome window — the label's own numerator), score a quick model, then delete it. The split is **grouped by client** so no client sits in both train and test.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def quick_auc(cols):
    X, y, g = frame[cols], frame.grows, frame.client
    tr, te = next(GroupShuffleSplit(1, test_size=0.3, random_state=0).split(X, y, g))
    m = HistGradientBoostingClassifier(random_state=0).fit(X.iloc[tr], y.iloc[tr])
    return roc_auc_score(y.iloc[te], m.predict_proba(X.iloc[te])[:, 1])

frame["LEAK_imp_late"] = frame.imp_late.fillna(0)           # <- deliberate leak
leaky = quick_auc(FEATS + ["LEAK_imp_late"])
print(f"WITH leak    (6 cols): AUC = {leaky:.3f}   <- suspiciously near perfect")
frame = frame.drop(columns=["LEAK_imp_late", "imp_late"])    # <- leak deleted
assert "imp_late" not in frame and "LEAK_imp_late" not in frame
honest = quick_auc(FEATS)
print(f"WITHOUT leak (5 cols): AUC = {honest:.3f}   <- the honest number I keep")

The jump toward 1.0 is not skill: `imp_late` is the label's numerator. The five-feature number is the one I report.

## 5) One named limitation of my slice

**Unbalanced panel / partial early windows.** Clients do not share a start date (`dim_clients.gsc_data_start`), and pages registered mid-March only accrue history from their registration day. Their "first 14 days" are partial and their growth label is mostly new-page ramp-up, not real opportunity. The ≥ 50 early-impressions filter only partly guards against this. I also rely on GSC alone: GA4 survives on a small share of rows (Query 3), so engagement is not in the model.

## 6) Self-check

- [x] Five plain-words contract answers (section 1)
- [x] Exactly three verification queries with outputs; availability uses `IS TRUE` (Query 3)
- [x] Five features, one "knowable because…" line each (section 3)
- [x] Deliberate leak added, score jump shown, leak deleted, honest score kept (section 4)
- [x] One named limitation (section 5)
- [x] Month 2026-03 only; June 2026 untouched; no token in the notebook